# 03: スキーマ進化（Schema Evolution）とドリフト耐性

## このノートブックで学習できること

- Day 2 の `pos_transactions` で上流レジ改修により新カラム（`payment_method`、`discount_amount`）が追加されても、パイプラインを停止せず `union_by_name`（Schema Merge）で自動追従する手順
- 型変換エラーを起こす異常値（`quantity = "two"` など）を `TRY_CAST` / `SAFE_CAST` 相当のロジックで検知し、データを破棄せず救済カラム（`_rescued_data` JSON 文字列）へ退避して取り込みを完走させる実装
- Iceberg の `field-id` ベース管理により、過去の Parquet ファイルを再書き込み（バックフィル）することなく `erp_products` のカラム名変更（`price` → `unit_price`）を行う検証

## 到達ゴール

- 列追加や型異常を含む上流データの変化（Schema Drift）に対して、ジョブを落とさず安全に取り込むパターンを実装できる
- カラムリネーム時に過去データファイルの書き直しが不要な理由を `field-id` の仕組みから説明できる

## なぜこれが必要か（従来のデータレイクとの比較）

従来のスキーマオンリード環境や厳格なロードジョブでは、上流に 1 列追加されたり 1 行だけ文字列が混入しただけで日次バッチ全体が失敗し、運用者の手動復旧が必要になっていました。また、カラム名を変更すると過去ファイルとの名前不一致が起きるため、全過去データの再作成（バックフィル）が必要でした。OTF の Schema Evolution と救済カラムを組み合わせることで、こうした運用停止を防げます。


## 1. Day 1 POS データの取り込みと Schema Merge（Day 2 の新カラム追従）

このセルでは、まず Day 1 の `pos_transactions.jsonl` に救済カラム `_rescued_data` を加えたスキーマで `bronze.pos_transactions` テーブルを作成して書き込みます。


In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.catalog_helper import (
    LANDING_DIR,
    LAKEHOUSE_DIR,
    get_duckdb_conn,
    get_iceberg_catalog,
    init_storage,
    register_iceberg_table,
)
import json
import polars as pl
import pyarrow as pa
from pyiceberg.schema import Schema
from pyiceberg.types import LongType, NestedField, StringType

catalog = get_iceberg_catalog()
if catalog.table_exists("bronze.pos_transactions"):
    catalog.drop_table("bronze.pos_transactions")

pos_schema_d1 = Schema(
    NestedField(1, "transaction_id", StringType(), required=True),
    NestedField(2, "store_id", StringType(), required=True),
    NestedField(3, "customer_id", StringType(), required=True),
    NestedField(4, "product_id", StringType(), required=True),
    NestedField(5, "quantity", LongType(), required=False),
    NestedField(6, "unit_price", LongType(), required=True),
    NestedField(7, "total_amount", LongType(), required=False),
    NestedField(8, "purchased_at", StringType(), required=True),
    NestedField(9, "_rescued_data", StringType(), required=False),
)
pos_tbl = catalog.create_table("bronze.pos_transactions", schema=pos_schema_d1)

d1_rows = [json.loads(line) for line in (LANDING_DIR / "day1" / "pos_transactions.jsonl").read_text(encoding="utf-8").splitlines()]
for r in d1_rows:
    r["_rescued_data"] = None
pos_tbl.append(pa.Table.from_pylist(d1_rows, schema=pos_tbl.schema().as_arrow()))
print(f"Day 1 取り込み完了: {pos_tbl.scan().to_arrow().num_rows} 行")


Day 1 の 300 件が書き込まれました。

次のセルでは、Day 2 の `pos_transactions.jsonl` を読み込みます。ここでは `quantity` を `SAFE_CAST` 相当の関数で整数へ変換し、変換に失敗した値（`"two"`, `"three"`, `"N/A"` など）は `quantity = None` にしたうえで元の値を `_rescued_data` に JSON 文字列として退避します。さらに、`update_schema().union_by_name(...)` で新カラム `payment_method` と `discount_amount` をテーブルスキーマへ自動マージして追記します。


In [ ]:
def parse_pos_with_rescue(jsonl_path: Path) -> list[dict]:
    parsed = []
    for line in jsonl_path.read_text(encoding="utf-8").splitlines():
        raw = json.loads(line)
        rescued = {}
        qty_raw = raw.get("quantity")
        try:
            raw["quantity"] = int(qty_raw)
        except (TypeError, ValueError):
            rescued["quantity"] = qty_raw
            raw["quantity"] = None
        raw["_rescued_data"] = json.dumps(rescued, ensure_ascii=False) if rescued else None
        parsed.append(raw)
    return parsed

d2_rows = parse_pos_with_rescue(LANDING_DIR / "day2" / "pos_transactions.jsonl")
d2_arrow_schema = pa.schema([
    pa.field("transaction_id", pa.string(), nullable=False),
    pa.field("store_id", pa.string(), nullable=False),
    pa.field("customer_id", pa.string(), nullable=False),
    pa.field("product_id", pa.string(), nullable=False),
    pa.field("quantity", pa.int64(), nullable=True),
    pa.field("unit_price", pa.int64(), nullable=False),
    pa.field("total_amount", pa.int64(), nullable=True),
    pa.field("purchased_at", pa.string(), nullable=False),
    pa.field("_rescued_data", pa.string(), nullable=True),
    pa.field("payment_method", pa.string(), nullable=True),
    pa.field("discount_amount", pa.int64(), nullable=True),
])
d2_arrow = pa.Table.from_pylist(d2_rows, schema=d2_arrow_schema)

with pos_tbl.update_schema() as update:
    update.union_by_name(d2_arrow.schema)

pos_tbl.append(d2_arrow)
print("進化後のスキーマ列:", [f.name for f in pos_tbl.schema().fields])

res_df = pl.from_arrow(pos_tbl.scan().to_arrow())
print(f"総行数: {res_df.height} / 救済された異常行数: {res_df.filter(pl.col('_rescued_data').is_not_null()).height}")
res_df.filter(pl.col("_rescued_data").is_not_null()).select(["transaction_id", "quantity", "payment_method", "_rescued_data"]).head(5)


パイプラインを停止させることなく、Day 2 の新カラム（`payment_method`, `discount_amount`）が自動追加され、型異常の 10 件も `_rescued_data` に元の値（例: `{"quantity": "two"}`）を残した状態で全 600 件が取り込まれました。Day 1 のデータには新カラムが `null` として自然に補完されます。

## 2. ゼロコピー・カラムリネーム（`erp_products` の `price` → `unit_price`）

次のセルでは、Day 1 の `erp_products.csv`（`price` 列）で Iceberg テーブル `bronze.erp_products` を作成し、そのデータファイル（Parquet）一覧を記録したあと、`update_schema().rename_column("price", "unit_price")` を実行します。その後、過去の Parquet ファイルが一切書き換わっていないことと、Day 2 の商品データ（`unit_price` 列）をそのまま追記できることを確認します。


In [ ]:
if catalog.table_exists("bronze.erp_products"):
    catalog.drop_table("bronze.erp_products")

prod_d1 = pl.read_csv(LANDING_DIR / "day1" / "erp_products.csv").to_arrow()
prod_tbl = catalog.create_table("bronze.erp_products", schema=prod_d1.schema)
prod_tbl.append(prod_d1)

data_dir = Path(prod_tbl.location().replace("file://", "")) / "data"
files_before = {p.name: p.stat().st_mtime_ns for p in data_dir.rglob("*.parquet")}

with prod_tbl.update_schema() as update:
    update.rename_column("price", "unit_price")

files_after_rename = {p.name: p.stat().st_mtime_ns for p in data_dir.rglob("*.parquet")}
print("リネーム前後で Parquet ファイルの更新時刻が一致するか:", files_before == files_after_rename)

prod_d2 = pl.read_csv(LANDING_DIR / "day2" / "erp_products.csv")
new_prods = prod_d2.filter(pl.col("product_id").is_in(["PRD-025", "PRD-026"]))
prod_tbl.append(pa.Table.from_pylist(new_prods.to_dicts(), schema=prod_tbl.schema().as_arrow()))

pl.from_arrow(prod_tbl.scan().to_arrow()).tail(5)


`files_before == files_after_rename` が `True` となり、過去の Parquet ファイルを 1 バイトも書き直さずにメタデータ上のカラム名変更だけで `price` が `unit_price` に切り替わりました。Day 2 で追加された `PRD-025`, `PRD-026` も同じ `unit_price` 列として統合されています。
